# Lab 2: Automatic Evaluation on Amazon Bedrock (LLM-as-Judge)

Without at least some level of **automated** testing and evaluation, it can be extremely challenging and time-consuming to iterate and optimize AI solutions: One change to a prompt template or inference parameter could have unexpected effects across multiple different scenarios, and re-testing every user journey by hand will be prohibitively slow. This becomes even more important when wanting to explore a broader range of models to optimize your solution.

On the other hand, since generative AI outputs are open-ended (text or content - not just a single number or label) they can be more complex to automatically evaluate than traditional software testing.

In this lab, we'll show how [LLM-as-a-Judge Evaluation Jobs on Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-judge.html) can help you run automated evaluation jobs to more efficiently test and compare different models for your use-case. As an example use-case, we'll evaluate LLM performance on **mathematical computations in the context of shopping**.

### How and Why LLM-as-Judge Evaluation Works

Since the outputs of Generative AI solutions are free-form (text or something else), interpreting a numeric score or pass/fail from them is non-trivial. In general, there'll usually be **multiple different aspects** of a free-form response that we could score: For example whether it was inoffensive, helpful, concise, or grounded in verifiable sources - are all separate and independent judgments.

There are many possible ways to approach these assessments automatically - including heuristic metrics from classical language processing or small models specialised for specific tasks like toxicity classifiers. Amazon Bedrock supports a range of these methods too, and you can explore some in the [optional extension notebook]() for this lab if you have time later.

However, one of the most powerful and flexible approaches is to simply **Use a 'judge' LLM to evaluate the output of your original/target LLM**:

- First, collect input prompt & output response data from your target LLM for your actual use-case
- Then, prompt another LLM to evaluate the outputs of the first one - according to some criteria you define, and outputting a simple score or pass/fail.

**This can work well even if your "judge" LLM is also not perfect, because** you can (and should!) usually frame the task of *checking* an answer to be cognitively simpler than *creating* one. For example, you might be asking your target LLM to answer a complex question based on information from multiple sources... But asking the judge LLM to just check "Is this answer: {...} in line with the ground truth: 42?".

We'll talk more later about best practices to boost the accuracy of your LLM-judged evaluations, and how Amazon Bedrock helps you adhere to these.

## Prerequisites
---

To use Automated Model Evaluation in Amazon Bedrock, you'll need to complete some pre-requisites.

### Summary of Required Resources and Permissions

In the following sections of the notebook we'll walk through setting up the required resources. Here's a summary of what we'll configure:

1. Amazon S3 Storage Configuration
    * Regional Compatibility: An Amazon S3 bucket must exist in the same AWS Region as your Amazon Bedrock models
        * Example: When using Bedrock in us-west-2, your S3 bucket must also be in us-west-2
    * CORS Configuration: The S3 bucket requires Cross Origin Resource Sharing (CORS) configuration enabled
        * This allows proper communication between Amazon Bedrock services and your storage.
2. IAM Role Requirements
The IAM role executing this notebook must have sufficient permissions to perform the following:
    * S3 Operations:
        * Read from and write to your designated Amazon S3 bucket
        * Upload evaluation datasets and retrieve results
    * Bedrock Service Access:
        * Invoke Amazon Bedrock foundation models
        * Create and manage model inference configurations
    * Evaluation Job Management:
       * Create and initiate evaluation jobs
       * Monitor job status and progress
       * Access and download evaluation results

For detailed documentation on the requirements and setup instructions for your own AWS account, please refer to the [official Amazon Bedrock documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-type-automatic.html).

### 🛠️ Environment Setup

If you haven't already, you'll need to install the required libraries for the workshop. See [../pyproject.toml](../pyproject.toml) for details.

We've commented-out the below cell because you should've already run the installation for [lab 1a](../lab1/Lab1a_-_Model_Selection_Framework.ipynb) - but if you need it, you can remove the `# ` and run the cell:

In [ ]:
# %pip install -e ..

> ⚠️ **Reminder**
> - You'll need to restart your notebook kernel (⤾ button in the toolbar above) *if you've already run* any of the code cells below before changing any library installations.
> - On SageMaker you might see an error the installation warning about conflicts due to pip's dependency resolution. As long as the installations themselves were successful, this can be ignored.

With the required Python libraries installed, we'll import them and connect to AWS services:

In [ ]:
# Python Built-Ins:
from datetime import datetime, timedelta, timezone
import json
import random
import re
import time

# External Dependencies:
import boto3  # AWS SDK for Python
from IPython.display import display, Markdown
import matplotlib.pyplot as plt  # Graph/chart plotting
import pandas as pd  # Tools for tabular data (dataframes)
import seaborn as sns  # Graph/chart plotting

botosess = boto3.Session()
region = botosess.region_name
print(f"Using region: {region}")

bedrock = botosess.client("bedrock")
iam = botosess.client("iam")
s3 = botosess.client("s3")

### Choose an S3 Bucket for Model Evaluation jobs

Bedrock model evaluation jobs require an Amazon S3 bucket in your current AWS region to store input datasets and model evaluation results.


**If you're running this notebook in the JupyterLab environment in Amazon SageMaker AI Studio, you can use your the default bucket from the Amazon SageMaker session to store the datasets and evaluation results. To do so, run the code below as-is.** 

For users running this notebook outside SageMaker AI Studio (for example on a local machine or EC2 instance), you'll need to either create a new S3 bucket or specify an existing one in your region. Please follow the instructions within the cell before you execute it.

In [ ]:
# Optionally manually configure your S3 bucket name (and a parent folder) here:
bucket = None
s3_prefix = "bedrock-evaluation-demo"

# Otherwise, we'll try to auto-discover a default one assuming you're in SageMaker:
if bucket is None:
    print("Finding SageMaker default bucket...")
    import sagemaker

    bucket = sagemaker.Session().default_bucket()

print(f"\nUsing S3 bucket:\ns3://{bucket}/{s3_prefix}")

### Enable Cross Origin Resource Sharing (CORS) on S3 bucket

The following CORS policy is only required on your S3 bucket for **human (manual)** evaluation jobs. Although this notebook will explore LLM-judged evaluation jobs so doesn't technically need it, we've included it so that your environment is set up to explore other types of Bedrock evaluation jobs too.

You can find more details on this requirement in the [here in the Amazon Bedrock User Guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-security-cors.html).

In [ ]:
cors_configuration = {
    "CORSRules": [
        {
            "AllowedHeaders": ["*"],
            "AllowedMethods": ["GET", "PUT", "POST", "DELETE"],
            "AllowedOrigins": ["*"],
            "ExposeHeaders": ["Access-Control-Allow-Origin"],
        }
    ]
}

s3.put_bucket_cors(Bucket=bucket, CORSConfiguration=cors_configuration)

### Provide an IAM service role

To grant the Amazon Bedrock evaluation job to take actions in your account (like accessing the source data from S3 and invoking foundation models), you'll need to provide it an **IAM service role** to assume.

You can find more details on service roles [here in the Bedrock User Guide](https://docs.aws.amazon.com/bedrock/latest/userguide/automatic-service-roles.html).

Below, we'll start by creating a new role that the Amazon Bedrock service is allowed to assume:

In [ ]:
aws_acct = botosess.client('sts').get_caller_identity().get('Account')

assume_role_policy_document = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowBedrockToAssumeRole",
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock.amazonaws.com"
            },
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {
                    "aws:SourceAccount": aws_acct
                },
                "ArnEquals": {
                    "aws:SourceArn": "arn:aws:bedrock:{}:{}:evaluation-job/*".format(
                        botosess.region_name, aws_acct
                    )
                }
            }
        }
    ]
})

In [ ]:
role_name="Amazon-Bedrock-model-eval-{}".format(str(datetime.now().timestamp()).split('.')[0])
create_role_response = iam.create_role(
    RoleName=role_name,
    AssumeRolePolicyDocument=assume_role_policy_document
)
role_arn = create_role_response["Role"]["Arn"]

# Wait for the role to propagate before we carry on...
waiter = iam.get_waiter('role_exists')
print(f"Waiting for role '{role_name}' to exist...")
waiter.wait(
    RoleName=role_name,
    WaiterConfig={
        'Delay': 2,        # Optional: Poll every 2 seconds instead of 1
        'MaxAttempts': 5,  # Optional: Max attempts 30 times instead of 20
    }
)
print(f"Role ready!\n{role_arn}")

#### Add permissions to access S3 data and Amazon Bedrock models

For the evaluation job to run, the role you pass will need permission to:
1. Access your input dataset and save output results in Amazon S3
2. Invoke the Foundation Models your job will use (including both the target model and the evaluator/judge model)

Below we'll define and attach some quite broad-scoped inline policies to the role to enable this. To customize these policies for your own AWS account and implement least-privilege access as a best practice, you can refer [here in the Bedrock User Guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-type-automatic.html) for more guidance.

In [ ]:
aws_s3_policy_doc = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowAccessToCustomDatasetsAndOutput",
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:ListBucket",
                "s3:PutObject"
            ],
            "Resource": [
                "arn:aws:s3:::{}".format(bucket),
                "arn:aws:s3:::{}/outputs/".format(bucket),
                "arn:aws:s3:::{}/custom_datasets/".format(bucket),
                "arn:aws:s3:::{}/*".format(bucket),
            ]
        }
    ]
})

aws_br_policy_doc = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AllowAccessToBedrockResources",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream",
                "bedrock:CreateModelInvocationJob",
                "bedrock:StopModelInvocationJob",
                "bedrock:GetProvisionedModelThroughput",
                "bedrock:GetInferenceProfile", 
                "bedrock:ListInferenceProfiles",
                "bedrock:GetImportedModel",
                "bedrock:GetPromptRouter",
                "bedrock:GetEvaluationJob",
                "bedrock:ListEvaluationJobs",
                "bedrock:CreateEvaluationJob",
                "sagemaker:InvokeEndpoint"
            ],
            "Resource": [
                "arn:aws:bedrock:*::foundation-model/*",
                "arn:aws:bedrock:*:{}:inference-profile/*".format(aws_acct),
                "arn:aws:bedrock:*:{}:provisioned-model/*".format(aws_acct),
                "arn:aws:bedrock:*:{}:imported-model/*".format(aws_acct),
                "arn:aws:bedrock:*:{}:application-inference-profile/*".format(aws_acct),
                "arn:aws:bedrock:*:{}:default-prompt-router/*".format(aws_acct),
                "arn:aws:sagemaker:*:{}:endpoint/*".format(aws_acct),
                "arn:aws:bedrock:*:{}:marketplace/model-endpoint/all-access".format(aws_acct)
            ]
        }
    ]
})

def wait_for_policy_propagation(iam_client, role_name, policy_name):
    for attempt in range(30):
        try:
            iam_client.get_role_policy(RoleName=role_name, PolicyName=policy_name)
            return
        except ClientError as e:
            if e.response['Error']['Code'] == 'NoSuchEntity':
                print("NoSuchEntity, trying again")
                time.sleep(1)
                continue
            raise

Next, we'll attach the policies to the created role and wait for them to become visible:

In [ ]:
iam_s3_response = iam.put_role_policy(
    RoleName=role_name,
    PolicyName="s3_access",
    PolicyDocument=aws_s3_policy_doc
)
iam_s3_response
wait_for_policy_propagation(iam, role_name, "s3_access")

In [ ]:
iam_bedrock_response = iam.put_role_policy(
    RoleName=role_name,
    PolicyName="br_access",
    PolicyDocument=aws_br_policy_doc
)
iam_bedrock_response
wait_for_policy_propagation(iam, role_name, "br_access")

Finally we'll store the details of the created role in case you continue on to the optional extension notebook later:

In [ ]:
%store role_arn
%store role_name

## Provide a dataset
---

With the infrastructure and permissions set up, it's time to define what you actually want to test.

For this example, we'll create a simple synthetic dataset of mathematical reasoning problems testing
1. Basic arithmetic
2. Logical reasoning, and
3. Natural language understanding

In [ ]:
def generate_shopping_problems(num_problems=50):
    """Generate shopping-related math problems with random values."""
    problems = []
    items = ["apples", "oranges", "bananas", "books", "pencils", "notebooks"]

    for _ in range(num_problems):
        # Generate random values
        item = random.choice(items)
        quantity = random.randint(3, 20)
        price_per_item = round(random.uniform(1.5, 15.0), 2)
        discount_percent = random.choice([10, 15, 20, 25, 30])

        # Calculate the answer
        total_price = quantity * price_per_item
        discount_amount = total_price * (discount_percent / 100)
        final_price = round(total_price - discount_amount, 2)

        # Create the problem
        problem = {
            "prompt": f"If {item} cost ${price_per_item} each and you buy {quantity} of them with a {discount_percent}% discount, how much will you pay in total?",
            "category": "Shopping Math",
            "referenceResponse": f"The total price will be ${final_price}. Original price: ${total_price} minus {discount_percent}% discount (${discount_amount})"
        }

        problems.append(problem)

    return problems


# Generate the sample dataset
SAMPLE_SIZE = 30
print(f"Generating dataset of {SAMPLE_SIZE} examples...")
problems = generate_shopping_problems(SAMPLE_SIZE)
print("✅ Done!")
print(f"\nExample item:\n{json.dumps(problems[0], indent=2)}")

We'll need to save the dataset in the [required format for Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-prompt-datasets-judge.html), which is based on [JSON Lines](https://jsonlines.org/):

In [ ]:
dataset_filename = "shopping_problems.jsonl"

with open(dataset_filename, "w") as f:
    for problem in problems:
        f.write(json.dumps(problem) + "\n")

display(Markdown(f"dataset written to [{dataset_filename}]({dataset_filename})"))

...And to actually use it in Amazon Bedrock, we'll need to upload the dataset to the Amazon S3 Bucket and folder prefix you configured earlier:

In [ ]:
dataset_s3key = "/".join((s3_prefix, dataset_filename))

try:
    s3.upload_file(dataset_filename, bucket, dataset_s3key)
    print(f"✓ Successfully uploaded to:\ns3://{bucket}/{dataset_s3key}")
except Exception as e:
    print(f"✗ ERROR uploading to S3: {str(e)}")
    raise e

## Configure and run the evaluation jobs

---

You're now ready to choose target models and configure the evaluation job(s).

For this example we'll create jobs to evaluate two different generator models on the "shopping math" problems dataset.

We'll enable the full range of [built-in evaluation metrics](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-metrics.html) to show all the different pre-built ways Amazon Bedrock can judge your generations. Note that [custom metric prompts are also supported](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-custom-metrics-prompt-formats.html), and can be a great way to further refine your evaluation criteria for your use case.

The cell below sets up a utility function we'll re-use later to start the multiple jobs:

In [ ]:
# Enable all available built-in LLM-as-judge metrics:
metric_names = [
    "Builtin.Correctness",
    "Builtin.Completeness", 
    "Builtin.Faithfulness",
    "Builtin.Helpfulness",
    "Builtin.Coherence",
    "Builtin.Relevance",
    "Builtin.FollowingInstructions",
    "Builtin.ProfessionalStyleAndTone",
    "Builtin.Harmfulness",
    "Builtin.Stereotyping",
    "Builtin.Refusal",
]


def create_llm_judge_evaluation(
    role_arn: str,
    input_s3_uri: str,
    output_s3_uri: str,
    generator_model_id: str,
    evaluator_model_id: str,
    dataset_name: str = None,
):
    # Eval jobs require full ARN for newer models
    generator_model_arn = f"arn:aws:bedrock:{region}::foundation-model/{generator_model_id}"
    evaluator_model_arn = f"arn:aws:bedrock:{region}::foundation-model/{evaluator_model_id}"

    # Construct a job name from the model IDs and current timestamp:
    job_name = "-".join((
        # Sanitized name of the generator model:
        generator_model_id.split(".")[-1].split(":")[0],
        # Provider of the evaluator model:
        evaluator_model_id.split(".")[0],
        # Timestamp:
        datetime.now().strftime('%Y-%m-%d-%H-%M-%S')
    ))
    job_name = re.sub(r'[^a-z0-9-]+', '-', job_name.lower())[:63]

    try:
        response = bedrock.create_evaluation_job(
            jobName=job_name,
            roleArn=role_arn,
            applicationType="ModelEvaluation",
            evaluationConfig={
                "automated": {
                    "datasetMetricConfigs": [
                        {
                            "taskType": "General",  # Must be 'General' for LLM-Judged
                            "dataset": {
                                "name": dataset_name or "CustomDataset",
                                "datasetLocation": {
                                    "s3Uri": input_s3_uri
                                }
                            },
                            "metricNames": metric_names,
                        }
                    ],
                    "evaluatorModelConfig": {
                        "bedrockEvaluatorModels": [
                            {
                                "modelIdentifier": evaluator_model_arn
                            }
                        ]
                    }
                }
            },
            inferenceConfig={
                "models": [
                    {
                        "bedrockModel": {
                            "modelIdentifier": generator_model_arn
                        }
                    }
                ]
            },
            outputDataConfig={
                "s3Uri": output_s3_uri
            }
        )

        return {
            "job_name": job_name,
            "job_arn": response["jobArn"],
            "generator_model": generator_model_id,
            "evaluator_model": evaluator_model_id,
            "status": "CREATED"
        }

    except Exception as e:
        print(f"ERROR creating evaluation job: {str(e)}")
        raise

### Choose models and start the jobs

We'll run evaluations on this dataset for two different target (generator) open weight models, to show how we can compare models against each other.

We'll follow the common practice of using a different and **larger** model as the evaluator (judge).

Since evaluation is usually less latency-sensitive and a lower overall volume than the actual use case traffic, it's often affordable to use a larger (and more generally-intelligent) model as the evaluator than your actual solution uses. A high-quality judge model can help to ensure your LLM-judged scores stay closely-aligned to human-reviewed scores - but in practice if your evaluation criteria are very clear and objective, it *may not be necessary*. Likewise when evaluation metric prompts are unclear or subjective, even using the smartest frontier models available will generally yield inconsistent results. The gold standard to check for selecting an evaluator model is whether it's delivering assessments that align well with what you(r human reviewers) think.

As an aside, it's also known that [models recognise and prefer their own generations](https://arxiv.org/abs/2404.13076). This is an important factor to consider especially if your judge LLM(s) are also in the list of generator models you're evaluating for your solution. Again, the best mitigation is to provide clear and objective evaluation criteria for the metric(s) you're asking the judge to measure: The less subjectivity is involved in the evaluation, the less that biases like these will affect your results.

In [ ]:
eval_model_id = "mistral.mistral-large-2402-v1:0"
gen_model_ids = [
    "nvidia.nemotron-nano-9b-v2",
    "mistral.mistral-7b-instruct-v0:2",
]

output_s3path = "/".join((s3_prefix, "model-eval-output"))

In [ ]:
eval_jobs = []
for gen_model_id in gen_model_ids:
    eval_jobs.append(
        create_llm_judge_evaluation(
            role_arn=role_arn,
            input_s3_uri=f"s3://{bucket}/{dataset_s3key}",
            output_s3_uri=f"s3://{bucket}/{output_s3path}",
            generator_model_id=gen_model_id,
            evaluator_model_id=eval_model_id,
        )
    )
    print(f"✓ Created job: {eval_jobs[-1]['job_name']}")
    print(f"  Generator: {gen_model_id}")
    print(f"  Evaluator: {eval_model_id}")
    print("-" * 60)
    time.sleep(1)
eval_job_arns = [j["job_arn"] for j in eval_jobs]

### Monitoring job status
The jobs will take several minutes to complete. You can monitor the progress of the evaluation jobs and display their current status before you proceed.

In [ ]:
if not eval_job_arns:
    print("⚠️ No evaluation jobs to monitor.")
else:
    for job_arn in eval_job_arns:
        try:
            response = bedrock.get_evaluation_job(jobIdentifier=job_arn)
            print(f"Job {response['jobName']}: Status {response['status']}")
        except Exception as e:
            print(f"ERROR fetching status for job {job_arn}:\n{e}")

You can also review the status of the jobs in the [Amazon Bedrock Console](https://console.aws.amazon.com/bedrock/home?#/eval/evaluation)

<div style="background-color: #d4edda; border-left: 4px solid #28a745; padding: 15px; border-radius: 5px;">

<strong>The evaluation jobs you just submitted may take several minutes to complete.</strong><br><br>


Instead of waiting for the submitted evaluation job(s) to complete, let's proceed with monitoring and analyzing results from previously completed jobs. This approach allows us to:

⏱️ Make productive use of our workshop time.

🧠 Understand the evaluation framework and metrics.

📈 Compare existing model performance results.

In the following cells, we'll:

🔄 Check the status of our submitted job(s).

📥 Retrieve and analyze results from completed evaluation jobs.

⚖️ Compare performance across different models.

📊 Visualize key metrics and insights.
</div>
<br/>
Next, we'll retrieve the most recent jobs completed for each generator, evaluator, and task type combination:

In [ ]:
def get_completed_llm_judge_jobs():
    all_jobs = []
    next_token = None

    # Get all jobs with pagination
    while True:
        params = {
            'sortBy': 'CreationTime',
            'sortOrder': 'Descending',
            'statusEquals': 'Completed',
            'applicationTypeEquals': 'ModelEvaluation',
            'maxResults': 1000
        }
        if next_token:
            params['nextToken'] = next_token

        response = bedrock.list_evaluation_jobs(**params)
        all_jobs.extend(response['jobSummaries'])
        next_token = response.get('nextToken')
        if not next_token:
            break

    # Filter jobs for LLM-as-judge evaluation
    jobs = [
        job for job in all_jobs
        if 'evaluatorModelIdentifiers' in job
        # and any(job.get('modelIdentifiers', []) == [model] for model in gen_models)
        # and job.get('evaluatorModelIdentifiers', []) == [eval_model]
    ]

    # Group jobs by unique combination of generator model and evaluator model
    job_groups = {}

    for job in jobs:
        generator_model = job['modelIdentifiers'][0]
        evaluator_model = job['evaluatorModelIdentifiers'][0]
        key = (generator_model, evaluator_model)

        # Keep only the most recent job for each unique combination
        if key not in job_groups or job['creationTime'] > job_groups[key]['creationTime']:
            job_groups[key] = job

    return list(job_groups.values())

In [ ]:
evaluation_jobs = get_completed_llm_judge_jobs()[:2]
evaluation_jobs

## Review evaluation results

---

Once you have some completed evaluation job(s), we can dive in to the results.

The first thing to note is that the [Amazon Bedrock Console](https://console.aws.amazon.com/bedrock/home?#/eval/evaluation) already offers an interactive UI for exploring job results - Click on a completed job name from the list to explore!

We can also download and explore the results programmatically, which we'll demonstrate here in the notebook. To get started, we'll first need to retrieve the output data locations on Amazon S3:

In [ ]:
job_output_jsonl_s3uris = []
for job in evaluation_jobs:
    job_arn = job["jobArn"]
    job_details = bedrock.get_evaluation_job(jobIdentifier=job_arn)
    job_name = job_details["jobName"]
    print(f"\nJob: {job_name}")

    # Parse output S3 URI
    # (Note that a folder named as the job is created under the configured location)
    job_output_s3uri = job_details['outputDataConfig']['s3Uri']
    if job_output_s3uri.endswith("/"):
        job_output_s3uri += job_name
    else:
        job_output_s3uri = "/".join((job_output_s3uri, job_name))
    job_output_bucket, _, job_output_prefix = job_output_s3uri[len("s3://"):].partition("/")

    # List objects
    response = s3.list_objects_v2(Bucket=job_output_bucket, Prefix=job_output_prefix)
    jsonl_files = [
        f"s3://{job_output_bucket}/{obj['Key']}"
        for obj in response.get('Contents', []) if obj['Key'].endswith('.jsonl')
    ]
    job_output_jsonl_s3uris.append(jsonl_files)
    print("Output files:")
    for f in jsonl_files:
        print(f"  {f}")

...and then read those data files:

In [ ]:
s3_res = boto3.resource('s3')

def read_jsonl_records(s3uris: str | list[str]) -> list[dict]:
    """Retrieve metrics from a Bedrock Evaluation Job output"""
    if isinstance(s3uris, str):
        s3uris = [s3uris]
    output_content = []
    for uri in s3uris:
        bucket, _, path = uri[len("s3://"):].partition("/")
        content_object = s3_res.Object(bucket, path)
        jsonl_content = content_object.get()['Body'].read().decode('utf-8')
        output_content.extend([json.loads(jline) for jline in jsonl_content.splitlines() if jline])
    return output_content

job_output_data = [read_jsonl_records(uris) for uris in job_output_jsonl_s3uris]

There's a lot of fine-grained detail in these output files - including scores and score explanations at the individual input prompt + output response level for each model tested.

Let's take a quick look at one example datum to see what's there:

In [ ]:
print(
    json.dumps(
        # First sample from the first evaluation job:
        job_output_data[0][0],
        indent=2,
    )
)

### Aggregate metrics

While the above individual example-level detail is useful for diving in to specific cases, we'd like to start out with higher-level visualizations to measure and compare the overall performance between models.

Run the cell below to extract all the jobs' output data into a unified table:

In [ ]:
def summarize_record(record: dict, **kwargs) -> dict:
    """Summarize an evaluation job output datum to just the model IDs and metric values"""
    result = {
        **kwargs,
        "generatorModel": record["modelResponses"][0]["modelIdentifier"],
        "evaluatorModel": None,
    }
    for score in record["automatedEvaluationResult"]["scores"]:
        result[score["metricName"]] = score["result"]
        evaluator_id = score["evaluatorDetails"][0]["modelIdentifier"]
        if result["evaluatorModel"] is None:
            result["evaluatorModel"] = evaluator_id
        else:
            if evaluator_id != result["evaluatorModel"]:
                raise ValueError(
                    "Unexpected mismatch: Multiple evaluator model IDs in one record. Got: %s"
                    % (result["evaluatorModel"], evaluator_id)
                )
    return result


# Extract datum summaries and flatten:
eval_output_flat = [
    summarize_record(
        r,
        jobName=job_detail["jobName"],
        sampleIndex=ix
    )
    for job_output, job_detail in zip(job_output_data, evaluation_jobs)
    for ix, r in enumerate(job_output)
]
# Convert from plain list to DataFrame table:
eval_output_df = pd.DataFrame(eval_output_flat)
print("Flattened data table (top 10 records):")
eval_output_df.head(10)

With this unified table, you can aggregate over the individual examples in each job to find the overall metrics that you care about.

For example, to look at the (mean) **average** for each score:

In [ ]:
eval_output_df.drop(columns="sampleIndex").groupby(
    ["jobName", "generatorModel", "evaluatorModel"]
).mean()

These same averages could be plotted on a chart:

In [ ]:
def plot_metric_summary(eval_output_df: pd.DataFrame):
    df_view = eval_output_df.drop(
        columns=["jobName", "evaluatorModel", "sampleIndex"]
    ).groupby(["generatorModel"]).mean().reset_index().melt(
        id_vars="generatorModel",
        var_name="metric",
        value_name="value",
    )
    fig = plt.figure(figsize=(32, 8))
    sns.set_style("whitegrid")
    sns.barplot(df_view, x="metric", y="value", hue="generatorModel")
    plt.title("Evalation Overview (Metrics Averaged Over Test Dataset)")
    return fig

summary_fig = plot_metric_summary(eval_output_df)

You can also check other aggregations beyond the average. 

For example, you might be interested in not just the average for `Builtin.ProfessionalStyleAndTone` but also the **minimum** - what was the score when each model performed at its worst, which could constitute a risk?

> ⚠️ **Remember:** For some metrics like `Builtin.Relevance` a higher score may be better - while for others like `Builtin.Harmfulness` a lower score is better. Review the [documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-metrics.html) on the built-in metrics, if you're not sure!

In [ ]:
min_professional_df = eval_output_df.groupby(
    ["jobName", "generatorModel", "evaluatorModel"]
).agg({"Builtin.ProfessionalStyleAndTone": ["min", "mean", "max"]})
min_professional_df

▶️ Review the metrics for the two generator models for this particular synthetic use case. Do they reflect what you'd expect, given any public intelligence benchmarks or leaderboards you can find - like [Artificial Analysis](https://artificialanalysis.ai/?models=nvidia-nemotron-nano-9b-v2-reasoning%2Cmistral-7b-instruct) for example?

### Plotting and comparing metric distributions

As well as aggregate summary metrics, we can consider directly plotting the distributions to get a better idea of how each models' scores vary between the dataset's examples.

For example, run the below to visualise the 'correctness' metric's variation for both models overlaid together:

In [ ]:
def plot_metric_histogram(eval_output_df: pd.DataFrame, metric: str, bins="auto"):
    if metric not in eval_output_df.columns:
        raise ValueError(f"Column '{metric}' is not in the given dataframe")
    metric_lower = metric.lower()
    if "harmfulness" in metric_lower or "toxicity" in metric_lower:
        sub = " (Lower the better)"
    elif "refusal" in metric_lower:
        sub = ""
    else:
        sub = " (Higher the better)"
    fig = plt.figure(figsize=(12, 6))
    sns.set_style("whitegrid")
    sns.histplot(data=eval_output_df, palette="flare", x=metric, hue="generatorModel", bins=bins)
    # plt.legend([f"{c} (avg {df_view[metric].mean():.2})" for c in df.columns], title='Model')
    plt.xlabel('Inference test')
    plt.ylabel('Number of samples')
    plt.title(f"{metric}{sub}")
    return fig


hist = plot_metric_histogram(eval_output_df, "Builtin.Correctness")

Your results may vary, but in this test we often see the correctness score distributions have clear peaks near 1 (correct), 0 (wrong), and 0.5 (mixed). Intermediate scores like 0.3 or 0.7 are relatively rare with this evaluator model and prompt on this example dataset.

It's quite common for LLM-judged metrics to cluster around specific values, rather than vary smoothly, because their [prompt guidance](https://docs.aws.amazon.com/bedrock/latest/userguide/model-evaluation-type-judge-prompt.html) usually defines discrete checks and encourages composing an aggregate score by summing those checks together. Asking a foundation model to simply generate a confidence score as a number in the output is generally fragile so usually should be avoided.

## Conclusions and takeaways

---

Although free-form generative AI outputs aren't as simple to categorize into "right" or "wrong" as traditional software unit testing, it's still important to automate evaluations because manual testing simply won't scale fast enough for you to experiment with different models, prompt templates, and other configurations to optimize and maintain your AI solutions.

Amazon Bedrock offers a range of [features to help with evaluation](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation.html), and in this notebook we saw [LLM-judged model evaluation jobs](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-judge.html) in action.

Using (judge) LLMs to evaluate the output of your main (generator) LLMs might seem worryingly circular at first, but it's a powerful and flexible technique so long as you keep a few key best practices in mind:

1. Keep your evaluation criteria **specific and objective**
    * The more complex or subjective of an assessment you ask your judge LLM to make, the greater the risk it could deviate from what a human reviewer would think.
    * Consider making separate judge LLM calls with independent prompt templates, to assess **different aspects** of the response independently and keep the "cognitive load" on the judge LLM low.
    * Be clear and consistent in your evaluation criteria about how the judge LLM should "measure" compliance. Asking the judge to assign continuous scores like 'confidence' is generally best avoided.
2. Consider your choice of judge LLM as compared to the target/generator model(s) you're evaluating
    * Remember if using the same model or same family of models for judge and target, that models are generally known to favour *their own* generations... But if evaluation criteria are specific and objective, this subjective bias may be insignificant.
    * It may be affordable to use a bigger, slower, more intelligent but costly model for evaluation than for the use-case itself - and this could help keep your automated evaluation scores more aligned to your human reviewers.
3. **Check** how well your automated scores align with human reviewers on the same examples, to understand the level and limitations of trust you can place in the automated scores.

Other types of evaluation job offered by Bedrock include:
- [Human-based model evaluation jobs](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-human.html) - to help orchestrate any human reviews you might want to perform to reconcile automated scores against
- [Knowledge Base evaluation jobs](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-kb.html) - with direct integration to and specialized metrics for Amazon Bedrock Knowledge Bases, to help evaluate RAG pipelines
- [Automatic (heuristic) model evaluation jobs](https://docs.aws.amazon.com/bedrock/latest/userguide/evaluation-automatic.html) - similar to the LLM-judged jobs we showed in this notebook, but using more classical NLP metrics and related techniques, rather than LLM-based evaluations.

## Next steps

If you have some extra time later, there's an [optional extension notebook](Lab2Extension_Classical_Metrics.ipynb) demonstrating automatic model evaluation jobs you can explore.

...But first, we'd suggest heading along to **[lab 3](../lab3/Lab3_Prompt_Optimization.ipynb)** to move beyond automatic *evaluation*, into automatically **optimizing** your solutions!